In [2]:
# STRAND NOTEBOOK INITIALISATION

# Core Inputs
import sys
from pathlib import Path
import os
import subprocess
import yaml

def find_project_root(markers=("pyproject.toml", ".git", "lib")) -> Path:
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if any((parent / marker).exists() for marker in markers):
            return parent
    raise RuntimeError(
        f"Could not find project root. Looked for markers {markers} "
        f"starting from {current}"
    )

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root detected at: {PROJECT_ROOT}")

# ----------------------------------------------------------
# 2️⃣ Load YAML config from config/
# ----------------------------------------------------------
CONFIG_PATH = PROJECT_ROOT / "config" / "project_config.yml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config file: {CONFIG_PATH}")

CFG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

print(f"⚙️  Project config loaded")

# ----------------------------------------------------------
# 3️⃣ Derive root prefix from git repo name
# ----------------------------------------------------------
def get_git_repo_root(path: Path) -> Path | None:
    try:
        out = subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "--show-toplevel"],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
        return Path(out).resolve()
    except Exception:
        return None


git_root = get_git_repo_root(PROJECT_ROOT)

yaml_prefix = (CFG.get("root_prefix") or "").strip("/")

if yaml_prefix:
    root_prefix = yaml_prefix
    print(f"📦 Using root_prefix from YAML: {root_prefix}")
else:
    root_prefix = git_root.name if git_root else PROJECT_ROOT.name
    print(f"📦 Using git repo name as root_prefix: {root_prefix}")



📁 Project root detected at: C:\Users\owyn8\Documents\Thesis\project-template\project-template
⚙️  Project config loaded
📦 Using git repo name as root_prefix: project-template


In [3]:
# Project Imports (ADD YOUR IMPORTS HERE)

import pandas as pd
import numpy as np

# Import more helper functions if necessary
from lib.helpers import (
    B2Config,
    make_s3_client,
    list_objects,
    join_key
)


# Set display defaults
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [4]:
# Initialise s3 session
cfg = B2Config.from_env(bucket_env="STRAND_BUCKET")
s3 = make_s3_client(cfg)
bucket = cfg.bucket

def s3_key(*parts: str) -> str:
    return join_key(root_prefix, *parts)

print(f"☁️  Connected to Backblaze bucket: {bucket}")
print(f"🔗 Endpoint: {cfg.endpoint_url}")

☁️  Connected to Backblaze bucket: STRAND2
🔗 Endpoint: https://s3.us-west-002.backblazeb2.com


In [7]:
paths = CFG["paths"]

inputs_raw_rel = paths["inputs"]["raw"]               # inputs/raw
inputs_processed_rel = paths["inputs"]["processed"]   # inputs/processed
outputs_raw_rel = paths["outputs"]["raw"]             # outputs/raw

INPUTS_RAW = s3_key(inputs_raw_rel)
INPUTS_PROCESSED = s3_key(inputs_processed_rel)
OUTPUTS_RAW = s3_key(outputs_raw_rel)

print("\n📦 Prefixes ready:")
print("  INPUTS_RAW       =", INPUTS_RAW)
print("  INPUTS_PROCESSED    =", INPUTS_PROCESSED)
print("  OUTPUTS_RAW         =", OUTPUTS_RAW)

params = CFG["params"]
print("Parameters Loaded:")
print(params)


📦 Prefixes ready:
  INPUTS_RAW       = project-template/inputs/raw
  INPUTS_PROCESSED    = project-template/inputs/processed
  OUTPUTS_RAW         = project-template/outputs/raw
Parameters Loaded:
None


In [6]:
print(f"\n🔎 Top-level under {root_prefix}:")
for obj in list_objects(s3, bucket, prefix=root_prefix, recursive=False):
    print("  -", obj["Key"])


🔎 Top-level under project-template:
